# 03 — Build the pipeline

One piece per cell. Test each on 2-3 cases before writing the next one.

Nothing here is a finished product — it's the workshop. Notebook 04 copies the working
functions out of here and runs them properly.

In [ ]:
import os, sys, json, re
from pathlib import Path

# works locally and in Colab
for candidate in (Path.cwd(), Path.cwd().parent, Path("/content/pa-appeal")):
    if (candidate / "data").exists():
        os.chdir(candidate)
        break
ROOT = Path.cwd()
print("working from:", ROOT)


In [ ]:
# Colab: put the key in the secrets panel (key icon on the left), named GEMINI_API_KEY
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except ImportError:
    pass   # locally: export GEMINI_API_KEY=... before launching jupyter

assert os.environ.get("GEMINI_API_KEY"), "no API key found"
print("key loaded")


In [ ]:
policies = {p.stem: p.read_text() for p in sorted(Path("data/policies").glob("*.md"))}
criteria  = json.load(open("data/criteria.json"))
cases     = json.load(open("data/cases.json"))
len(policies), len(criteria), len(cases)


In [ ]:
import hashlib, time

CACHE = Path("data/cache"); CACHE.mkdir(parents=True, exist_ok=True)

# Check AI Studio for what your project actually has. Flash-Lite models get the biggest
# free daily allowance; the bigger Flash models get far fewer calls per day.
MODEL = "gemini-flash-lite-latest"

def ask(prompt, system="", model=MODEL, force=False):
    """Ask the LLM, but only once per unique prompt. Repeats come off disk.

    This is the single most useful thing on a free tier. Restart & Run All costs
    zero API calls once the cache is warm.
    """
    key = hashlib.sha256(f"{model}|{system}|{prompt}".encode()).hexdigest()[:16]
    f = CACHE / f"{key}.json"
    if f.exists() and not force:
        return json.loads(f.read_text())["response"]

    from google import genai   # SDK surface changes - check current docs if this errors
    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

    for attempt in range(5):
        try:
            r = client.models.generate_content(
                model=model,
                contents=prompt,
                config={"system_instruction": system,
                        "temperature": 0,
                        "response_mime_type": "application/json"},
            )
            text = r.text
            break
        except Exception as e:
            if "429" not in str(e) or attempt == 4:
                raise
            wait = 5 * 2 ** attempt
            print(f"rate limited, waiting {wait}s")
            time.sleep(wait)

    f.write_text(json.dumps({"model": model, "system": system,
                             "prompt": prompt, "response": text}, indent=2))
    return text

def ask_json(prompt, system="", **kw):
    raw = ask(prompt, system, **kw)
    return json.loads(re.sub(r"^```(json)?|```$", "", raw.strip(), flags=re.M))


## Chunking

Dumb version and smart version. The smart one exists so a two-part rule never gets cut in
half — that's the single biggest improvement in the whole pipeline.

In [ ]:
def chunk_fixed(text, doc_id, size=512, overlap=64):
    words, step, out = text.split(), size - overlap, []
    for i in range(0, max(1, len(words)), step):
        w = words[i:i + size]
        if not w:
            break
        out.append({"id": f"{doc_id}::fix::{len(out)}", "doc": doc_id, "text": " ".join(w),
                    "criterion": None})
    return out

def chunk_headings(text, doc_id, min_chars=200):
    marks = list(re.finditer(r"^#{1,6}\s+(.*)$", text, re.M))
    if not marks:
        return chunk_fixed(text, doc_id)
    out = []
    for n, m in enumerate(marks):
        end = marks[n + 1].start() if n + 1 < len(marks) else len(text)
        body = text[m.start():end].strip()
        if len(body) < min_chars and out:
            out[-1]["text"] += "\n\n" + body
            continue
        out.append({"id": f"{doc_id}::sec::{len(out)}", "doc": doc_id, "text": body,
                    "criterion": None})
    return out

def chunk_by_criteria(criteria, policies):
    out = [{"id": f"crit::{c['id']}", "doc": c["source"], "text": c["text"], "criterion": c["id"]}
           for c in criteria if c["text"]]
    for doc_id, text in policies.items():
        if doc_id != "L33718":
            out.extend(chunk_headings(text, doc_id))
    return out

fixed  = [c for d, t in policies.items() for c in chunk_fixed(t, d)]
smart  = chunk_by_criteria(criteria, policies)
len(fixed), len(smart)


**Check the decoys are actually in there.** If a number later looks too good, this is the first thing to look at.

In [ ]:
from collections import Counter
Counter(c["doc"] for c in smart)


## Search

~200 chunks, so the "vector database" is one numpy array and a dot product.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")   # CPU is fine

def build_index(chunks):
    texts = [c["text"] for c in chunks]
    M = embedder.encode(texts, normalize_embeddings=True)
    from rank_bm25 import BM25Okapi
    bm25 = BM25Okapi([t.lower().split() for t in texts])
    return {"chunks": chunks, "M": np.asarray(M, dtype=np.float32), "bm25": bm25}

def _minmax(a):
    lo, hi = a.min(), a.max()
    return np.zeros_like(a) if hi - lo < 1e-9 else (a - lo) / (hi - lo)

def search(query, index, mode="dense", top_k=5, w=0.5):
    q = embedder.encode([query], normalize_embeddings=True)[0]
    dense = index["M"] @ q
    if mode == "dense":
        scores = dense
    else:
        # normalize each first - cosine and BM25 are on totally different scales
        scores = w * _minmax(dense) + (1 - w) * _minmax(np.asarray(index["bm25"].get_scores(query.lower().split())))
    order = np.argsort(scores)[::-1][:top_k]
    return [{**index["chunks"][i], "score": float(scores[i])} for i in order]


## Reranker

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("BAAI/bge-reranker-base")

def rerank(query, hits, top_k=5):
    scores = reranker.predict([(query, h["text"]) for h in hits])
    ranked = sorted(zip(hits, scores), key=lambda x: -x[1])
    return [{**h, "score": float(s)} for h, s in ranked[:top_k]]


## Ask the model

The one line that matters: absent information is insufficient_evidence, not unmet.

In [ ]:
SYSTEM = """You decide whether a patient record satisfies Medicare coverage criteria.

Rules:
1. Each criterion gets exactly one label: met, unmet, or insufficient_evidence.
2. ABSENT INFORMATION IS insufficient_evidence, NOT unmet.
3. evidence_quote must be copied character-for-character from the policy text provided.
4. If no supporting quote is in the provided text, use insufficient_evidence and leave
   evidence_quote empty.
5. reasoning: at most two sentences.

Return JSON: {"decisions": [{"criterion_id", "label", "evidence_quote",
"source_doc_id", "reasoning"}]}
"""

def decide(case, retrieved, criteria_asked):
    policy_text = "\n\n---\n\n".join(
        f"[{c['doc']}] {c['text']}" for c in retrieved)
    docs = case["documents"]
    prompt = (f"POLICY TEXT:\n{policy_text}\n\n"
              f"CRITERIA TO DECIDE:\n{criteria_asked}\n\n"
              f"SLEEP STUDY:\n{docs['sleep_study']}\n\n"
              f"CHART NOTE:\n{docs['chart_note']}\n\n"
              f"DENIAL LETTER:\n{docs['denial_letter']}")
    return ask_json(prompt, SYSTEM)["decisions"]


## Check the quotes

Not AI. String matching. This is the number nobody can argue with.

In [ ]:
import unicodedata
from rapidfuzz import fuzz

_SUBS = {"\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"',
         "\u2013": "-", "\u2014": "-", "\u00a0": " "}

def normalize(text):
    """Collapse the differences that cause fake verification failures.

    Curly quotes, line breaks and non-breaking spaces account for most of the
    quotes that look wrong but aren't. Always normalize before blaming the model.
    """
    text = unicodedata.normalize("NFKC", text or "")
    for bad, good in _SUBS.items():
        text = text.replace(bad, good)
    return re.sub(r"\s+", " ", text).strip().casefold()

def check_quote(quote, source_text, all_docs=None, threshold=95):
    """-> 'exact' | 'close' | 'wrong_doc' | 'made_up' | 'empty'"""
    q = normalize(quote)
    if not q:
        return "empty"
    src = normalize(source_text)
    if q in src:
        return "exact"
    if fuzz.partial_ratio(q, src) >= threshold:
        return "close"
    for other in (all_docs or {}).values():
        o = normalize(other)
        if q in o or fuzz.partial_ratio(q, o) >= threshold:
            return "wrong_doc"
    return "made_up"

VERIFIED = ("exact", "close")

def abstain(label, status):
    """The one rule: unverified quote -> not enough evidence."""
    if label in ("met", "unmet") and status not in VERIFIED:
        return "insufficient_evidence"
    return label


Try it on one case end to end:

In [ ]:
index = build_index(smart)
case = cases[0]
hits = search("AHI events per hour sleep study", index, mode="hybrid", top_k=5)
for d in decide(case, hits, "B1: AHI/RDI at least 15 with at least 30 events"):
    status = check_quote(d["evidence_quote"], policies.get(d["source_doc_id"], ""), policies)
    print(d["criterion_id"], d["label"], "->", status, "->", abstain(d["label"], status))
